# Projeto G1 — Análise do Consumo de Água no Brasil

## Disciplina: Linguagem de Programação — Análise e Visualização de Dados com Python

**Professor:** Alexandre Neves Louzada  
**Aluna:** Ysis Barbosa Gomes

Este notebook apresenta a etapa de análise exploratória e preparação dos dados do projeto sobre o consumo de água no Brasil.

O objetivo é analisar os padrões de consumo de água entre 2015 e 2024, utilizando indicadores, gráficos e interpretações que posteriormente serão utilizados em um dashboard interativo desenvolvido com Streamlit.

# 1. Problema de análise

A água é um recurso essencial para o abastecimento urbano, agricultura, indústria, geração de energia e saúde pública. Com o crescimento populacional, as mudanças climáticas e o desperdício, torna-se importante compreender como ocorre o consumo de água nas diferentes regiões do Brasil.

A análise busca responder:

- Quais estados apresentam maior consumo de água?
- Existe sazonalidade no consumo?
- Quais setores apresentam maior consumo?
- Há regiões mais vulneráveis à escassez?
- Existe relação entre estiagem e consumo?
- Houve aumento do consumo de água ao longo do tempo?
- Quais estados apresentam os maiores índices de desperdício?

Essas perguntas orientam toda a análise.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sqlalchemy import create_engine

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# 2. Leitura da base de dados

In [ ]:
df = pd.read_csv("simulacao_consumo_agua_brasil.csv")

df.head()

# 3. Check-up inicial da base

In [ ]:
print("Dimensões da base:", df.shape)

print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes:")
print(df.isnull().sum())

print("\nDuplicidades:")
print(df.duplicated().sum())

# 4. Preparação dos dados

Nesta etapa, os dados são preparados para facilitar as análises posteriores.

A coluna de data será convertida para o formato de data do Pandas e serão criadas novas variáveis para auxiliar na análise temporal e na avaliação do desperdício de água.

In [ ]:
df["data"] = pd.to_datetime(df["data"], errors="coerce")

df["ano_mes"] = df["data"].dt.to_period("M").astype(str)
df["trimestre"] = df["data"].dt.quarter

df["desperdicio_milhoes_litros"] = (
    df["consumo_milhoes_litros"] *
    df["desperdicio_percentual"] / 100
)

df.head()

In [ ]:
print("Período analisado:")
print(df["data"].min(), "até", df["data"].max())

print("\nRegiões:")
print(df["regiao"].unique())

print("\nSetores:")
print(df["setor_consumo"].unique())

print("\nNíveis de alerta:")
print(df["nivel_alerta"].unique())

# 5. KPIs principais

Os KPIs foram escolhidos para fornecer uma visão geral sobre o consumo de água no Brasil:

- Consumo total de água;
- Estado com maior consumo;
- Setor com maior consumo;
- Consumo per capita médio;
- Desperdício médio;
- Nível médio dos reservatórios.

In [ ]:
consumo_total = df["consumo_milhoes_litros"].sum()

consumo_uf = (
    df.groupby("uf")["consumo_milhoes_litros"]
    .sum()
    .sort_values(ascending=False)
)

estado_maior_consumo = consumo_uf.index[0]
valor_maior_consumo = consumo_uf.iloc[0]

consumo_setor = (
    df.groupby("setor_consumo")["consumo_milhoes_litros"]
    .sum()
    .sort_values(ascending=False)
)

setor_maior_consumo = consumo_setor.index[0]
valor_setor_maior_consumo = consumo_setor.iloc[0]

consumo_per_capita_medio = df["consumo_per_capita"].mean()
desperdicio_medio = df["desperdicio_percentual"].mean()
reservatorio_medio = df["reservatorios_percentual"].mean()

In [ ]:
print(f"Consumo total: {consumo_total:,.2f} milhões de litros")

print(
    f"Estado com maior consumo: {estado_maior_consumo} "
    f"({valor_maior_consumo:,.2f} milhões de litros)"
)

print(
    f"Setor com maior consumo: {setor_maior_consumo} "
    f"({valor_setor_maior_consumo:,.2f} milhões de litros)"
)

print(f"Consumo per capita médio: {consumo_per_capita_medio:,.2f}")

print(f"Desperdício médio: {desperdicio_medio:.2f}%")

print(f"Nível médio dos reservatórios: {reservatorio_medio:.2f}%")

# 6. Evolução do consumo ao longo do tempo

Nesta etapa, será analisada a evolução do consumo de água ao longo do período entre 2015 e 2024.

Os dados serão agrupados por mês para permitir a identificação de possíveis tendências de crescimento, queda ou variações no consumo ao longo do tempo.

In [ ]:
serie_mensal = (
    df.groupby("ano_mes")["consumo_milhoes_litros"]
    .sum()
    .reset_index()
    .sort_values("ano_mes")
)

serie_mensal.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    serie_mensal["ano_mes"],
    serie_mensal["consumo_milhoes_litros"]
)

ax.set_title("Evolução Mensal do Consumo de Água no Brasil")
ax.set_xlabel("Período")
ax.set_ylabel("Consumo (milhões de litros)")

ax.tick_params(axis="x", rotation=45)

# Exibe apenas alguns períodos no eixo X
intervalo = 12
ax.set_xticks(serie_mensal.index[::intervalo])
ax.set_xticklabels(
    serie_mensal["ano_mes"].iloc[::intervalo]
)

plt.tight_layout()
plt.show()

In [ ]:
consumo_anual = (
    df.groupby("ano")["consumo_milhoes_litros"]
    .sum()
    .reset_index()
    .sort_values("ano")
)

consumo_anual

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.lineplot(
    data=consumo_anual,
    x="ano",
    y="consumo_milhoes_litros",
    marker="o",
    ax=ax
)

ax.set_title("Consumo Anual de Água no Brasil")
ax.set_xlabel("Ano")
ax.set_ylabel("Consumo (milhões de litros)")

ax.set_xticks(consumo_anual["ano"])

plt.tight_layout()
plt.show()

### Interpretação

A análise do consumo anual mostra oscilações ao longo do período entre 2015 e 2024, sem uma tendência contínua de crescimento.

O maior consumo anual ocorreu em 2019, enquanto 2023 apresentou o menor valor do período analisado. Em 2024, observa-se novamente um aumento em relação ao ano anterior.

Portanto, os dados não indicam um crescimento constante do consumo de água ao longo dos anos, mas sim variações entre os períodos analisados.

# 7. Consumo de água por região

Nesta etapa, será analisado o consumo total de água nas cinco regiões brasileiras.

Essa comparação permite identificar quais regiões concentram os maiores volumes de consumo de água durante o período analisado.

In [ ]:
consumo_regiao = (
    df.groupby("regiao")["consumo_milhoes_litros"]
    .sum()
    .reset_index()
    .sort_values("consumo_milhoes_litros", ascending=False)
)

consumo_regiao

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=consumo_regiao,
    x="regiao",
    y="consumo_milhoes_litros",
    ax=ax
)

ax.set_title("Consumo Total de Água por Região")
ax.set_xlabel("Região")
ax.set_ylabel("Consumo (milhões de litros)")

plt.tight_layout()
plt.show()

### Interpretação

A Região Sudeste apresenta o maior consumo acumulado de água no período analisado, seguida pelas regiões Nordeste e Sul.

As regiões Centro-Oeste e Norte apresentam os menores volumes de consumo total dentro da base analisada.

Esses resultados mostram diferenças importantes no consumo entre as regiões brasileiras. Entretanto, o consumo total de uma região não deve ser interpretado isoladamente como maior ou menor eficiência no uso da água, pois fatores como população, quantidade de estados e características dos setores de consumo também devem ser considerados.

# 8. Consumo de água por setor

Nesta etapa, será analisado o consumo total de água entre os diferentes setores presentes na base de dados.

A comparação permite identificar quais setores concentram os maiores volumes de consumo durante o período analisado.

In [ ]:
consumo_por_setor = (
    df.groupby("setor_consumo")["consumo_milhoes_litros"]
    .sum()
    .reset_index()
    .sort_values("consumo_milhoes_litros", ascending=False)
)

consumo_por_setor

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=consumo_por_setor,
    x="setor_consumo",
    y="consumo_milhoes_litros",
    ax=ax
)

ax.set_title("Consumo Total de Água por Setor")
ax.set_xlabel("Setor")
ax.set_ylabel("Consumo (milhões de litros)")

plt.tight_layout()
plt.show()

### Interpretação

O setor Agrícola apresenta o maior consumo acumulado de água no período analisado, com aproximadamente 19.256 milhões de litros.

O setor Residencial aparece logo em seguida, com aproximadamente 19.212 milhões de litros, apresentando uma diferença pequena em relação ao setor Agrícola.

Os setores Comercial, Industrial e Público aparecem posteriormente no ranking. Apesar das diferenças observadas, o consumo encontra-se relativamente distribuído entre os cinco setores presentes na base.

Portanto, embora o setor Agrícola ocupe a primeira posição, os dados não apresentam uma concentração extremamente elevada do consumo em apenas um setor.

# 9. Consumo de água por UF

Nesta etapa, será analisado o consumo total de água por Unidade Federativa (UF).

O objetivo é identificar quais estados apresentam os maiores volumes de consumo durante o período analisado e comparar as diferenças existentes entre eles.

In [ ]:
consumo_por_uf = (
    df.groupby("uf")["consumo_milhoes_litros"]
    .sum()
    .reset_index()
    .sort_values("consumo_milhoes_litros", ascending=False)
)

consumo_por_uf

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

sns.barplot(
    data=consumo_por_uf,
    x="consumo_milhoes_litros",
    y="uf",
    ax=ax
)

ax.set_title("Ranking de Consumo de Água por UF")
ax.set_xlabel("Consumo (milhões de litros)")
ax.set_ylabel("UF")

plt.tight_layout()
plt.show()

### Interpretação

O Rio de Janeiro apresenta o maior consumo acumulado de água entre as UFs presentes na base, com aproximadamente 10.176 milhões de litros.

São Paulo, Minas Gerais e Espírito Santo aparecem nas posições seguintes, fazendo com que os quatro primeiros colocados pertençam à Região Sudeste.

A Bahia aparece na quinta posição e é o primeiro estado fora da Região Sudeste entre os cinco maiores consumidores.

Os resultados reforçam a maior participação da Região Sudeste no consumo total observado anteriormente. Entretanto, os valores representam o consumo acumulado da base e não devem ser interpretados isoladamente como indicadores de eficiência ou desperdício.

# 10. Sazonalidade do consumo de água

Nesta etapa, será analisado o comportamento do consumo de água ao longo dos meses do ano.

O objetivo é verificar se determinados meses apresentam padrões recorrentes de maior ou menor consumo, indicando uma possível sazonalidade.

In [ ]:
consumo_mensal_medio = (
    df.groupby("mes")["consumo_milhoes_litros"]
    .mean()
    .reset_index()
)

consumo_mensal_medio

In [ ]:
meses = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Abr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Set",
    10: "Out",
    11: "Nov",
    12: "Dez"
}

consumo_mensal_medio["nome_mes"] = (
    consumo_mensal_medio["mes"].map(meses)
)

consumo_mensal_medio

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.lineplot(
    data=consumo_mensal_medio,
    x="mes",
    y="consumo_milhoes_litros",
    marker="o",
    ax=ax
)

ax.set_title("Consumo Médio de Água por Mês")
ax.set_xlabel("Mês")
ax.set_ylabel("Consumo médio (milhões de litros)")

ax.set_xticks(range(1, 13))
ax.set_xticklabels(
    ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
     "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
)

plt.tight_layout()
plt.show()

In [ ]:
consumo_ano_mes = (
    df.groupby(["ano", "mes"])["consumo_milhoes_litros"]
    .sum()
    .reset_index()
)

tabela_heatmap = consumo_ano_mes.pivot(
    index="ano",
    columns="mes",
    values="consumo_milhoes_litros"
)

tabela_heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(
    tabela_heatmap,
    annot=True,
    fmt=".0f",
    ax=ax
)

ax.set_title("Sazonalidade do Consumo de Água")
ax.set_xlabel("Mês")
ax.set_ylabel("Ano")

ax.set_xticklabels(
    ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
     "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
)

plt.tight_layout()
plt.show()

### Interpretação

A análise mensal permite observar variações no consumo de água entre os diferentes meses e anos.

O mapa de calor facilita a identificação dos períodos de maior e menor consumo ao longo da série histórica. Apesar das oscilações observadas, não se identifica visualmente um padrão sazonal extremamente forte e constante durante todo o período.

Dessa forma, os dados indicam variações mensais no consumo, mas essas diferenças devem ser analisadas em conjunto com outros fatores, como chuva, temperatura e nível dos reservatórios.

# 11. Relação entre chuva e consumo de água

Nesta etapa, será analisada a relação entre o volume de chuva e o consumo de água.

O objetivo é verificar se períodos com maiores ou menores volumes de chuva apresentam alguma relação com o consumo registrado na base de dados.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.scatterplot(
    data=df,
    x="chuva_mm",
    y="consumo_milhoes_litros",
    alpha=0.5,
    ax=ax
)

ax.set_title("Relação entre Chuva e Consumo de Água")
ax.set_xlabel("Volume de chuva (mm)")
ax.set_ylabel("Consumo (milhões de litros)")

plt.tight_layout()
plt.show()

In [ ]:
correlacao_chuva_consumo = df[
    "chuva_mm"
].corr(
    df["consumo_milhoes_litros"]
)

print(
    f"Correlação entre chuva e consumo: "
    f"{correlacao_chuva_consumo:.4f}"
)

### Interpretação

O gráfico de dispersão não apresenta um padrão evidente entre o volume de chuva e o consumo de água.

A correlação de Pearson entre as variáveis é aproximadamente zero, indicando ausência de uma relação linear relevante entre chuva e consumo dentro da base analisada.

Portanto, os dados não fornecem evidências suficientes para afirmar que períodos com maior ou menor volume de chuva estão diretamente associados a mudanças no consumo de água.

É importante destacar que esse resultado se refere à base analisada e não significa que fatores climáticos não possam influenciar o consumo de água em situações reais.

# 12. Análise do desperdício de água

Nesta etapa, será analisado o percentual médio de desperdício de água entre as Unidades Federativas.

O objetivo é identificar quais estados apresentam os maiores índices de desperdício dentro da base analisada.

In [ ]:
desperdicio_por_uf = (
    df.groupby("uf")["desperdicio_percentual"]
    .mean()
    .reset_index()
    .sort_values(
        "desperdicio_percentual",
        ascending=False
    )
)

desperdicio_por_uf

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

sns.barplot(
    data=desperdicio_por_uf,
    x="desperdicio_percentual",
    y="uf",
    ax=ax
)

ax.set_title("Desperdício Médio de Água por UF")
ax.set_xlabel("Desperdício médio (%)")
ax.set_ylabel("UF")

plt.tight_layout()
plt.show()

In [59]:
top_5_desperdicio = desperdicio_por_uf.head(5)

top_5_desperdicio

,uf,desperdicio_percentual
2,CE,32.307792
3,DF,31.768000
0,AM,31.316750
12,PE,30.748500
13,PR,30.690167


### Interpretação

A análise evidencia diferenças nos percentuais médios de desperdício entre as Unidades Federativas presentes na base.

O ranking permite identificar os estados que apresentam os maiores índices médios de desperdício, complementando a análise de consumo realizada anteriormente.

É importante observar que um estado com alto consumo total não necessariamente apresenta o maior percentual de desperdício. Dessa forma, consumo e desperdício representam indicadores diferentes e devem ser analisados separadamente.

# 13. Nível dos reservatórios por região

Nesta etapa, será analisado o nível médio dos reservatórios nas diferentes regiões brasileiras.

O objetivo é comparar a disponibilidade hídrica entre as regiões presentes na base de dados.

In [ ]:
reservatorios_regiao = (
    df.groupby("regiao")["reservatorios_percentual"]
    .mean()
    .reset_index()
    .sort_values(
        "reservatorios_percentual",
        ascending=False
    )
)

reservatorios_regiao

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=reservatorios_regiao,
    x="regiao",
    y="reservatorios_percentual",
    ax=ax
)

ax.set_title("Nível Médio dos Reservatórios por Região")
ax.set_xlabel("Região")
ax.set_ylabel("Nível médio dos reservatórios (%)")

plt.tight_layout()
plt.show()

### Interpretação

Os níveis médios dos reservatórios apresentam variações entre as regiões analisadas.

A comparação permite identificar quais regiões apresentam maiores ou menores níveis médios de armazenamento de água dentro da base.

Como os dados utilizados são simulados, os resultados representam apenas o comportamento da base analisada e não devem ser interpretados como um retrato da situação hídrica real do Brasil.

# 14. Análise de correlação

Nesta etapa, será analisada a correlação entre algumas das principais variáveis numéricas da base.

O objetivo é verificar a intensidade das relações lineares entre consumo, desperdício, nível dos reservatórios, chuva, temperatura, população e consumo per capita.

In [ ]:
colunas_numericas = [
    "consumo_milhoes_litros",
    "desperdicio_percentual",
    "reservatorios_percentual",
    "chuva_mm",
    "temperatura_media",
    "populacao",
    "consumo_per_capita"
]

matriz_correlacao = df[colunas_numericas].corr()

matriz_correlacao

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sns.heatmap(
    matriz_correlacao,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=ax
)

ax.set_title("Matriz de Correlação das Variáveis")

plt.tight_layout()
plt.show()

### Interpretação

A matriz de correlação indica que o consumo de água apresenta correlações lineares muito baixas com as demais variáveis analisadas.

Não foram identificadas relações lineares fortes entre o consumo e fatores como chuva, temperatura, nível dos reservatórios, desperdício, população ou consumo per capita.

Esse resultado indica apenas a ausência de correlações lineares relevantes na base analisada e não significa que essas variáveis não possam apresentar outros tipos de relação.

# 15. Persistência com SQLAlchemy e SQLite

Nesta etapa, os dados tratados serão armazenados em um banco de dados SQLite utilizando SQLAlchemy.

A persistência dos dados permite armazenar a base de forma estruturada e realizar consultas SQL diretamente no banco de dados.

In [64]:
engine = create_engine(
    "sqlite:///consumo_agua.sqlite"
)

df.to_sql(
    "consumo_agua",
    engine,
    if_exists="replace",
    index=False
)

print("Tabela consumo_agua criada no banco SQLite.")

Tabela consumo_agua criada no banco SQLite.


### Consulta ao banco de dados

A seguir, será realizada uma consulta SQL para verificar o consumo total de água por região utilizando os dados armazenados no SQLite.

In [65]:
consulta_regiao = pd.read_sql(
    """
    SELECT
        regiao,
        SUM(consumo_milhoes_litros) AS consumo_total
    FROM consumo_agua
    GROUP BY regiao
    ORDER BY consumo_total DESC
    """,
    engine
)

consulta_regiao

,regiao,consumo_total
0,Sudeste,32497.94
1,Nordeste,20252.66
2,Sul,15145.77
3,Centro-Oeste,12371.52
4,Norte,12298.32


# 16. Conclusão

A análise dos dados permitiu identificar diferentes padrões relacionados ao consumo de água no Brasil entre 2015 e 2024.

Entre as regiões analisadas, o Sudeste apresentou o maior consumo acumulado de água. Na análise por Unidade Federativa, o Rio de Janeiro ocupou a primeira posição no ranking de consumo, seguido principalmente por outros estados da Região Sudeste.

Na comparação entre os setores, o setor Agrícola apresentou o maior consumo acumulado, seguido de perto pelo setor Residencial. A análise temporal mostrou oscilações ao longo dos anos, sem indicar uma tendência contínua de crescimento do consumo durante todo o período.

A análise de sazonalidade mostrou variações entre os meses e anos, enquanto a relação entre chuva e consumo apresentou correlação linear próxima de zero. A matriz de correlação também não indicou relações lineares fortes entre o consumo e as demais variáveis analisadas.

Também foram observadas diferenças nos percentuais de desperdício entre as Unidades Federativas e nos níveis médios dos reservatórios entre as regiões.

Por fim, os dados tratados foram armazenados em um banco SQLite utilizando SQLAlchemy, permitindo a realização de consultas SQL sobre a base.

Como os dados utilizados neste projeto são simulados, os resultados representam padrões existentes na base analisada e não devem ser interpretados como um diagnóstico da situação hídrica real do Brasil.